# 진단 — 평균 덮어쓰기가 어텐션 쪽 값을 망가뜨리는가

**어느 스텝·어느 RQ:** step3·step5(RQ2·RQ3)의 **측정 방법 자체를 점검**한다. 새 결과를 만드는 실험이 아니다.

**무엇을 확인하나.**
우리는 "얼마나 보는지"(Key)와 "무엇을 읽는지"(Value)를 각각 바꿔치기해 비교했다. 그런데 **Key에는
그 조각이 문장 몇 번째에 있었는지가 회전 형태로 박혀 있고(RoPE), Value에는 없다.** 서로 다른 위치의
조각을 평균 내면 회전 방향이 엇갈려 **서로 지워진다.** 그러면 우리가 집어넣는 건 정상 크기의 값이
아니라 **쪼그라든 값**이다.

→ "어텐션을 바꿔도 안 돌아온다"가 **어텐션 경로의 성질**인지 **우리가 값을 망가뜨린 탓**인지 갈라야 한다.
편향의 방향이 우리 결론과 같으므로 반드시 확인해야 하는 항목이다.

**어떻게 재나.**

    줄어듦 = ‖조각들의 평균‖ ÷ 조각 노름들의 평균

- 1에 가까움 → 조각들이 같은 방향을 봤다. 평균 내도 안 줄었다 = **문제 없음**
- 많이 작음 → 서로 지워졌다 = **쪼그라든 값을 넣고 있었다**

**Key만 재면 판단할 수 없다.** 서로 다른 조각을 평균 내면 회전이 없어도 조금은 줄어든다.
그래서 **Value의 같은 값을 기준선으로 함께 잰다.**

**결과가 어느 쪽으로 나오든 무슨 뜻인지 미리 밝힌다.**

| 나오는 그림 | 뜻 | 다음 |
|---|---|---|
| 어텐션 줄어듦 ≈ 내용 줄어듦 | 회전 때문에 손해 본 것이 없다 | **재실험 불필요.** 이 수치를 논문 부록에 싣고 "어텐션 경로가 정보를 안 나른다"를 확정 |
| 어텐션 줄어듦 ≪ 내용 줄어듦 | 우리가 어텐션 쪽 값을 망가뜨리고 있었다 | 위치를 맞춰 어텐션 조건만 봉우리 층 근방 재측정 |

**부하.** 점수를 매기지 않는다 — 프롬프트를 몇 번 통과시키고 나머지는 산수다.
**모델 하나에 몇 분.** 무료 T4로 충분하다. (deepseek-6.7b만 메모리가 빠듯할 수 있다.)

**모델 하나씩.** 셀 ④의 `PICK`을 바꿔 네 번 돌린다. 끊겨도 저장된 건 건너뛴다.

In [ ]:
# ② 환경 — 설치, GPU 확인, 무작위값 42 고정
!pip install -q -r requirements.txt

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (매우 느림)')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('무작위값 고정:', SEED)
print('transformers/torch 버전은 결과 meta에 자동 기록된다')


In [ ]:
# ③ 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
BRANCH = 'integration/step1-5'
!git fetch --quiet origin $BRANCH
!git checkout $BRANCH
!git pull --quiet origin $BRANCH
!pip install -e . -q
import sys; sys.path.insert(0, 'src')
print('브랜치:', BRANCH)


In [ ]:
# ④ 조건 설정 — 모델 하나 고르고, 코드 쪽·지침 쪽 둘 다 진단한다
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',           family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-6.7b-instruct',  family='deepseek',  dtype='float16'),
    ModelSpec(name='unsloth/Llama-3.2-3B-Instruct',             family='llama',     dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',       family='stability', dtype='float16'),
]

# ★ 이번에 돌릴 모델 하나 (0=qwen, 1=deepseek, 2=llama, 3=stable)
PICK = 0
MODEL = MODELS[PICK]
print('이번 모델:', MODEL.family, '|', MODEL.name)

# 진단 대상 두 갈래 — step3(코드 이름)와 step5(지침 지시어)가 같은 치환 코드를 쓴다.
#   코드 쪽 : 위치가 어긋난다(문맥 길이가 달라짐) → 회전 어긋남이 클 것으로 예상
#   지침 쪽 : 위치가 거의 안 어긋난다(지시어 시작 위치가 같음) → 작을 것으로 예상
BLOCKS = list(range(6))       # 42묶음 중 앞 6개면 충분하다(값이 묶음마다 크게 안 변한다)

conditions = []
for block in BLOCKS:
    pre = PrecedingCode(n_compliant=6, n_functions=12, composition=Composition.POOL, pool_block=block)
    for target_word, notation in (('code', Notation.CAMEL), ('instruction', Notation.CAMEL),
                                  ('instruction', Notation.SNAKE)):
        conditions.append(Condition(
            model=MODEL, preceding=pre,
            instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=notation),
            intervention=Intervention(kind=InterventionKind.KEY_VALUE, layers='sweep',
                                      donor=('compliant' if target_word == 'code' else 'opposite'),
                                      target=target_word),
            seed=SEED, token_unit='mean'))
print('조건 수:', len(conditions), '(묶음', len(BLOCKS), '× 3갈래)')


In [ ]:
# ⑤ 실행 — 조건마다 즉시 저장(재개). 점수를 안 매기므로 빠르다.
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model

STEP = 'diag_kv-phase'
todo = [c for c in conditions if not result_path(c, step=STEP).exists()]
print(f'[{MODEL.family}] 전체 {len(conditions)} / 남은 {len(todo)}')

if todo:
    handle = load_model(MODEL)
    print(f'  층수 {handle.num_layers}')
    for i, c in enumerate(todo, 1):
        out = run(c, handle=handle, mode='kv_diagnose')
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ2/RQ3(방법 점검)',
                                 prediction='어텐션 줄어듦이 내용 줄어듦보다 뚜렷이 작으면 방법 편향'))
        ex = out.metrics.extra
        mid = sorted(out.metrics.per_layer)[len(out.metrics.per_layer)//2]
        v = out.metrics.per_layer[mid]
        print(f"  [{i}/{len(todo)}] 대상={ex['intervention_target']:<11} "
              f"위치 어긋남 평균={ex['position_offset_abs_mean']:.1f}칸 | "
              f"중간층 줄어듦: 어텐션 {v['key_shrink']:.3f} / 내용 {v['value_shrink']:.3f}")
    del handle
    import torch, gc; gc.collect(); torch.cuda.empty_cache()
print('완료')


In [ ]:
# ⑥ 결과 로드
from harness import result_path
from harness.results import load_result
recs = [load_result(result_path(c, step='diag_kv-phase')) for c in conditions
        if result_path(c, step='diag_kv-phase').exists()]
print('불러온 조건:', len(recs))


In [ ]:
# ⑦ 요약 — 층별로 '어텐션 줄어듦'과 '내용 줄어듦'을 나란히 본다
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt

by_target = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
offsets = defaultdict(list)
for r in recs:
    t = r.metrics.extra['intervention_target']
    offsets[t].append(r.metrics.extra['position_offset_abs_mean'])
    for L, v in r.metrics.per_layer.items():
        by_target[t][int(L)]['key'].append(v['key_shrink'])
        by_target[t][int(L)]['value'].append(v['value_shrink'])

print('덮을 자리와 공여 자리의 위치 어긋남(칸):')
for t, xs in offsets.items():
    print(f'  {t:<12} 평균 {np.mean(xs):.2f}칸')

print()
print(f"{'대상':<12}{'층':>5}{'어텐션 줄어듦':>14}{'내용 줄어듦':>13}{'차이':>9}")
for t in by_target:
    layers = sorted(by_target[t])
    for L in layers[::max(1, len(layers)//8)]:
        k = np.mean(by_target[t][L]['key']); v = np.mean(by_target[t][L]['value'])
        print(f'{t:<12}{L:>5}{k:>14.3f}{v:>13.3f}{k-v:>9.3f}')

fig, axes = plt.subplots(1, len(by_target), figsize=(5.5*len(by_target), 4), squeeze=False)
for ax, t in zip(axes[0], by_target):
    layers = sorted(by_target[t])
    ax.plot(layers, [np.mean(by_target[t][L]['key']) for L in layers],
            label='Key (has position rotation)', color='tab:gray')
    ax.plot(layers, [np.mean(by_target[t][L]['value']) for L in layers],
            label='Value (no rotation) = baseline', color='tab:blue')
    ax.axhline(1.0, color='black', linewidth=0.7, linestyle=':')
    ax.set_ylim(0, 1.05); ax.set_xlabel('Layer'); ax.set_ylabel('Shrink ratio after averaging')
    ax.set_title(f'target = {t}'); ax.legend(fontsize=8); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print()
print('읽는 법: 회색이 파랑과 붙어 있으면 → 회전 때문에 손해 본 것이 없다 (재실험 불필요).')
print('         회색만 뚜렷이 아래면 → 어텐션 쪽 값을 우리가 망가뜨리고 있었다 (위치 맞춰 재측정).')


In [ ]:
# ⑧ 결과 폴더 zip 다운로드
import shutil
shutil.make_archive('diag_kv-phase_results', 'zip', 'results/diag_kv-phase')
try:
    from google.colab import files; files.download('diag_kv-phase_results.zip')
except Exception as e:
    print('Colab 아님(로컬):', e)
